In [1]:
import numpy as np
import csv
import pandas as pd
from itertools import islice
import json
import matplotlib.pyplot as plt
pd.set_option("display.max_rows", None)


In [2]:
books = pd.read_csv('data/final_book_dataset_cleaned.csv', sep = '\t',
        dtype={
            'author_birthyear' : 'Int64',
            'title_id' : 'Int64',
            'isbn' : 'str'
        }
    )

In [3]:
books.head(5)

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,tags,hugo,locus
0,9372,The Long Loud Silence,Wilson Tucker,1954,1954-00-00,Dell,1914,"Deer Creek, Illinois, USA",0899683754,Publisher's description: Tomorrow's war -- the...,"['Anatomy of Wonder 1 Core Collection', 'biolo...",False,False
1,2215181,Mrs. Candy Strikes It Rich,Robert Tallant,1954,1954-00-00,Doubleday,1909,"New Orleans, Louisiana, USA",NaN,NaN,NaN,False,False
2,1392436,Return to the Lost Planet,Angus MacVicar,1954,1954-00-00,Burke,1908,"Duror, Argyll, Scotland, UK",NaN,NaN,NaN,False,False
3,1112206,Rainbow on the Road,Esther Forbes,1954,1954-00-00,Houghton Mifflin,1891,"Westborough, Massachusetts, USA",NaN,NaN,NaN,False,False
4,1908,The Forgotten Planet,Murray Leinster,1954,1954-00-00,Ace Books,1896,"Norfolk, Virginia, USA",0881846163,"**From the first page of the Ace Double:** ""Na...","['insects', 'Librivox', 'Project Gutenberg', '...",False,False


In [4]:
## add month of publication column, 00 means only year is known
books['month_of_publication'] =books['release_date'].str[5:7].str.replace('00','Unknown')
books.sample(50)

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,tags,hugo,locus,month_of_publication
30782,19906,Darkest Hour,Mark Chadbourn,2000,2000-10-00,Gollancz,1960,"Ashby-de-la-Zouch, Leicestershire, England, UK",0575069031,NaN,['fantasy'],False,False,10
54559,1223179,Cornerstonia: The Rose River Journey,V. M. Rosario,2010,2010-10-25,Cornerstonia,<NA>,NaN,9780982858806,NaN,NaN,False,False,10
143359,3575942,Ghost Therapy,Mark Rosendorf,2024,2024-10-02,The Wild Rose Press,<NA>,NaN,9781509258239.0,NaN,['young-adult ghost story'],False,False,10
72599,3376269,Mona Livelong: Paranormal Detective,Valjeanne Jeffers,2013,2013-10-25,Valjeanne Jeffers,1967,USA,9781493591008.0,NaN,NaN,False,False,10
145569,3351766,The Witch of Wol Sin Lake,Lena Jeong,2024,2024-10-29,Harper,<NA>,NaN,9780063241701.0,After her fraught journey to save her queendom...,"['Fantasy', 'Young Adult']",False,False,10
50272,1268517,The Fight for the Frozen Land: Arctic,Elizabeth Singer Hunt,2009,2009-02-05,Red Fox,1970,"Roanoke, Virginia, USA",9781862306332,NaN,['juvenile sf'],False,False,02
17152,1113563,Spring-Heeled Jack,Philip Pullman,1989,1989-00-00,Alfred A. Knopf,1946,"Norwich, Norfolk, England, UK",0679810579,Three children make their escape from a London...,"['Fantasy', 'Young Adult', 'Adventure']",False,False,Unknown
30209,791721,The PowerBook,Jeanette Winterson,2000,2000-09-00,Alfred A. Knopf,1959,"Manchester, Lancashire, England, UK",0375411119,NaN,NaN,False,False,09
29038,19877,The Fata Morgana,Leo Frankowski,1999,1999-08-00,Baen Books,1943,"Detroit, Michigan, USA",0671578227,NaN,['lost race'],False,False,08
78147,1758585,Jerry's Magic,W. W. Rowe,2014,2014-09-07,Larson Publications,1934,NaN,9781936012664.0,NaN,['juvenile fantasy'],False,False,09


In [5]:
## Add age of author at publication
books['Author_Age_at_Publication'] = 'Unknown'
books['Author_Age_at_Publication'] = books['Author_Age_at_Publication'].case_when([(books['author_birthyear'].isna() == False,books['release_year'] - books['author_birthyear'])])

In [6]:
books.sample(15)

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,tags,hugo,locus,month_of_publication,Author_Age_at_Publication
57165,1254317,Unforsaken,Sophie Littlefield,2011,2011-10-11,Delacorte Press,<NA>,NaN,9780385738545,NaN,['young-adult fantasy'],False,False,10,Unknown
73444,1711689,The Lovely and the Lost,Page Morgan,2014,2014-05-01,Hot Key Books,<NA>,NaN,9781471402555.0,NaN,"['demons', 'gargoyles', 'historical fantasy', ...",False,False,05,Unknown
103118,3345048,Wisteria Wrinkle,Angela Pepper,2018,2018-07-28,Angela Pepper Publishing,<NA>,NaN,9781777672768.0,NaN,NaN,False,False,07,Unknown
104911,2485399,For Love of Evil,Auryn Hadley,2018,2018-11-21,Spotted Horse Productions,<NA>,NaN,9781731166180.0,NaN,NaN,False,False,11,Unknown
43818,1230457,The Sibyl's Urn,John T. Cullen,2007,2007-05-01,Clocktower Books,1949,NaN,9780743309066,NaN,NaN,False,False,05,58
144772,3316106,Grim Root,Bonnie Jo Stufflebeam,2024,2024-06-04,Dark Matter Ink,<NA>,NaN,9781958598368.0,NaN,['horror'],False,False,06,Unknown
22261,19853,Jane Eyre,Charlotte Brontë,1994,1994-00-00,Wordsworth Editions,1816,"Thornton, West Riding of Yorkshire, England, UK",1853268348,"""The orphaned Jane Eyre is no beauty but her p...","['1001 Books You Must Read Before You Die', 'h...",False,False,Unknown,178
31181,22071,The Dastard,Piers Anthony,2000,2000-10-00,Tor,1934,"Oxford, Oxfordshire, England, UK",0812574737,NaN,['fantasy'],False,False,10,66
139051,3385620,Of Bones and Skulls,Samantha Ziegler,2023,2023-04-26,Finished Fiction,<NA>,NaN,9798987953419.0,NaN,NaN,False,False,04,Unknown
100477,2132305,Future Threat,Elizabeth Briggs,2017,2017-03-01,AW Teen,<NA>,NaN,9780807526842.0,Elena Martinez survived the future. But the fi...,"['dystopia', 'time travel', 'young-adult sf', ...",False,False,03,Unknown


In [7]:
## Add column with number of hugo or locus awards won prior to that date
books['Hugo_Awards_Previously']= books.groupby(['author'])['hugo'].cumsum()
books['Locus_Awards_Previously']= books.groupby(['author'])['locus'].cumsum()

In [8]:
books.sample(15)

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,tags,hugo,locus,month_of_publication,Author_Age_at_Publication,Hugo_Awards_Previously,Locus_Awards_Previously
13518,10680,Galaxy Four,William Emms,1985,1985-11-00,W. H. Allen,1930,Australia,0491036914,NaN,NaN,False,False,11,55,0,0
116937,2619045,Cole Blooded,"Outspan Foster, Blaise Corvin",2019,2019-09-03,Adom Publishing,<NA>,NaN,9781690662235.0,NaN,NaN,False,False,09,Unknown,0,0
83148,1914366,An Explorer's Guide to the Nether,Winter Morgan,2015,2015-11-03,Sky Pony Press,<NA>,NaN,9781510703513.0,NaN,"['juvenile sf', 'video games']",False,False,11,Unknown,0,0
23201,12437,Scorpianne,Emily Devenport,1994,1994-08-00,Roc,1958,NaN,0451453182,On Earth Lucy has long been one of the best in...,[],False,False,08,36,0,0
74099,2644606,Gilded Wings,Cameo Renae,2014,2014-12-18,Crushing Hearts and Black Butterfly Publishing,<NA>,"San Francisco, California, USA",9781939769718.0,NaN,NaN,False,False,12,Unknown,0,0
109515,2346734,The Beast's Heart,Leife Shallcross,2018,2018-04-24,Hodder & Stoughton,<NA>,NaN,9781473668713.0,NaN,"['fantasy', 'retold fairy tales']",False,False,04,Unknown,0,0
108657,2367371,Whiskey on the Rocks,S. M. Blooding,2018,2018-04-02,Whistling Book Press,<NA>,NaN,9781947790209.0,NaN,['urban fantasy'],False,False,04,Unknown,0,0
37012,155434,The Certification of America: The Decline and ...,Paul Seifert,2004,2004-05-00,iUniverse,<NA>,NaN,0595314988,NaN,NaN,False,False,05,Unknown,0,0
79733,2239585,Corrigan Lust,Helen Harper,2015,2015-12-27,Helen Harper,<NA>,"Scotland, UK",NaN,NaN,"['fantasy', 'magic', 'paranormal', 'romance', ...",False,False,12,Unknown,0,0
119605,2753478,Maelstrom,Sonia Orin Lyris,2020,2020-07-14,Knotted Road Press,<NA>,NaN,9781644701614.0,NaN,['fantasy'],False,False,07,Unknown,0,0


In [9]:
## Add column with boolean if author been nominated for Hugo/Locus prior to that date
books['Hugo_Nominee_Before'] = 'False'
books['Hugo_Nominee_Before'] = books['Hugo_Nominee_Before'].case_when([(books['Hugo_Awards_Previously'] > 0,True)])
books['Locus_Nominee_Before'] = 'False'
books['Locus_Nominee_Before'] = books['Locus_Nominee_Before'].case_when([(books['Locus_Awards_Previously'] > 0,True)])

In [10]:
books.sample(100)

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,tags,hugo,locus,month_of_publication,Author_Age_at_Publication,Hugo_Awards_Previously,Locus_Awards_Previously,Hugo_Nominee_Before,Locus_Nominee_Before
79733,2239585,Corrigan Lust,Helen Harper,2015,2015-12-27,Helen Harper,<NA>,"Scotland, UK",NaN,NaN,"['fantasy', 'magic', 'paranormal', 'romance', ...",False,False,12,Unknown,0,0,False,False
118522,3038584,Blood Brothers,R. L. King,2020,2020-06-15,Magespace Press,<NA>,NaN,9781734096156.0,Alastair Stone figures using tracking magic to...,['Fantasy'],False,False,06,Unknown,0,0,False,False
44214,222991,Conversations with the Devil,Jeff Rovin,2007,2007-03-00,Forge,1951,"Brooklyn, New York, USA",0765307030,NaN,[],False,False,03,56,0,0,False,False
5946,2006971,The Stronghold,Mollie Hunter,1974,1974-00-00,Hamish Hamilton,1922,"Longniddry, East Lothian, Scotland, UK",0241890268,"Crippled in a Roman raid on his native island,...",['druids'],False,False,Unknown,52,0,0,False,False
112402,2529235,Legend,Nicole Conway,2019,2019-05-07,Month9Books,<NA>,"Decatur, Alabama, USA",9781948671361.0,"In a war of gods and tyrants, the will of the ...",['young-adult fantasy'],False,False,05,Unknown,0,0,False,False
60530,1352114,Angels Cry,J. S. Wayne,2011,2011-09-07,Noble Romance Publishing,1977,"Amarillo, Texas, USA",9781605927077,NaN,NaN,False,False,09,34,0,0,False,False
42826,180662,Engaging the Enemy,Elizabeth Moon,2006,2006-00-00,Del Rey / Ballantine,1945,"McAllen, Texas, USA",0345447565,Kylara Vatta has had to leave a glowing future...,"['female main character', 'ofearna-ebooks', 's...",False,False,Unknown,61,1,3,True,True
96332,2210451,Rules for Thieves,Alexandra Ott,2017,2017-06-06,Aladdin,<NA>,NaN,9781481472746.0,NaN,"['curses', 'juvenile fantasy']",False,False,06,Unknown,0,0,False,False
68814,2932089,Sam & Skully,Dewey Badeaux,2013,2013-00-00,Alligator Press,<NA>,NaN,9780988405745.0,"""Sam McDonnell isn't like the other kids in sc...",['Texas'],False,False,Unknown,Unknown,0,0,False,False
2692,1011452,Claimed,Francis Stevens,1966,1966-00-00,Renaissance E Books,1883,"Minneapolis, Minnesota, USA",1588731804,"**From the flaps of the Avalon edition:** ""It ...","['New Jersey', 'pgas', 'prehistoric god', 'sci...",False,False,Unknown,83,0,0,False,False


In [11]:
## Add Author Birthplace by country
replace_dict = {'Austria':['Austria-Hungary','Austro-Hungarian Empire','Austria'],
               'Russia':['Russian Empire','Russia','USSR','Soviet'],
               'Germany':['German Empire','Reich','Confederation','Germany','Prussia'],
               'Nigeria':['Nigeria'],
               'India':['INdia','India'],
               'Peru':['Perú','Peru'],
               'Ceylon':['Ceylon'],
               'Papua New Guinea':['Netherlands New Guinea'],
               'Hong Kong':['Territory','Kong','Hong Kong','Komg'],
               'Dominican Republic':['Dominican Republic'],
               'Bailiwack of Jersey':['Balliwack of Jersey','Bailiwick of Jersey'],
               'Trinidad and Tobago':['Tobago','Trinidad'],
               'UK': ['Yorkshire','Wales','England','United Kingdom','UK','Scotland','Northern Ireland','Guernsey','Britain'],
               'Australia':['Sydney','New South Wales','Australia','Queensland','Victoria','Gold Coast'],
               'USA':['Philadelphia','MO','PA','WA','PR','Baltimore','USA','US Virgin Islands','Puerto Rico'],
               'Panama':['Canal Zone','Panama'],
               'Sri Lanka': ['Lanka'],
               'New Zealand':['Zealand','NZ'],
               'East Indies' :['East Indies'],
                'West Indies' :['West Indies','Antilles','Dominica'],
                'El Salvador' :['El Salvador'],
                'South Africa' :['Cape Colony','Pretoria','Port Elizabeth','South Africa','Orange Free'],
                'United Arab Emirates' :['United Arab Emirates'],
                'Sierra Leone' :['Sierra Leone'],
                'Canada': ['British North America','Canada','Alberta'],
                'Guyana':['Guyana','British Guiana'],
                'Cayman Islands': ['Cayman'],
                'Marshall Islands':['Marshall'],
                'Thailand': ['Thailand','Siam'],
                'Malaysia':['Malaya','Malaysia'],
                'Philipines':['Philipines','Philippine Islands'],
                'Ireland' : ['Ireland','Irish'],
                'Turkey':['Turkey','Ottoman Empire'],
                'Rhodesia':['Rhodesia'],
                'Bermuda':['Bermuda'],
                'Kenya':['Kenya'],
                'Jamaica':['Jamaica'],
                'Sweden':['Sweden','Swedish'],
                'Singapore':['Singapore'],
                'Nyasaland':['Nyasaland'],
                'Palestine':['Palestine'],
                'Fiji':['Fiji'],
                'Holy Roman Empire':['Holy Roman Empire'],
                'Myanmar':['Burma','Myanmar'],
                'Malta':['Malta'],
                'Czech Republic':['Czech'],
                'Rwanda':['Rwanda','Portugese West Africa','Kamundongo']
}

def replace_author_birthplace_country(x):
    if pd.isna(x):
        return 'Unknown'
    for country, birthplace in replace_dict.items():
        if any(word in x for word in birthplace):
            return country
    if x not in replace_dict.items():
        return x.strip().split()[-1]
    return x



In [12]:
books = books.assign(author_birthplace_country=books['author_birthplace'].apply(replace_author_birthplace_country))
books.sample(15)

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,tags,hugo,locus,month_of_publication,Author_Age_at_Publication,Hugo_Awards_Previously,Locus_Awards_Previously,Hugo_Nominee_Before,Locus_Nominee_Before,author_birthplace_country
84259,1862200,Hostage,"Sherwood Smith, Rachel Manija Brown",2015,2015-01-06,Book View Café,1951,"Glendale, California, USA",9781611384734.0,NaN,[],False,False,01,64,0,0,False,False,USA
47638,714945,Grimspace,Ann Aguirre,2008,2008-02-26,Ace Books,1970,USA,9780441015993,"As the carrier of a rare gene, Sirantha Jax ha...","['adventure', 'ofearna-2008', 'ofearna-ebooks'...",False,False,02,38,0,0,False,False,USA
29292,20024,The Stainless Steel Rat Joins the Circus,Harry Harrison,1999,1999-11-00,Tor,1925,"Stamford, Connecticut, USA",0312869347,When you are the richest man in the Universe a...,"['circus', 'science fiction', 'thief', 'Scienc...",False,False,11,74,2,2,True,True,USA
37849,916020,Attack of the Bounty Hunters,Steven Gordon,2004,2004-00-00,Clifford Croft Press,<NA>,NaN,0974965200,NaN,NaN,False,False,Unknown,Unknown,0,0,False,False,Unknown
63547,1523606,Flora's Fury,Ysabeau S. Wilce,2012,2012-05-08,Harcourt / Houghton Mifflin Harcourt,<NA>,"California, USA",9780152054090,_Publisher:_ Determined to find her true mothe...,['young-adult fantasy'],False,False,05,Unknown,0,0,False,False,USA
982,1528063,Providence Island,Jacquetta Hawkes,1959,1959-04-00,Random House,1910,"Cambridge, Cambridgeshire, England, UK",NaN,"Lost race novel, set on a Pacific Island.\n\n",['lost race'],False,False,04,49,0,0,False,False,UK
29139,19503,Foundation's Triumph,David Brin,1999,1999-05-00,HarperPrism,1950,"Glendale, California, USA",0061052418,The Second Foundation Trilogy begins with Greg...,"['cyborgs', 'Earth', 'interstellar travel', 'p...",False,False,05,49,6,8,True,True,USA
80305,11380,The Interloper,John Russell Fearn,2015,2015-03-31,Gateway / Orion,1908,"Worsley, Lancashire, England, UK",9781473210318.0,NaN,"['armaments baron', 'interplanetary relations']",False,False,03,107,0,0,False,False,UK
139124,3149746,Rebel Squad,"Michael Anderle, Martha Carr",2023,2023-03-06,LMBPN Publishing,<NA>,NaN,9798885419550.0,NaN,NaN,False,False,03,Unknown,0,0,False,False,Unknown
137150,3088691,A Consuming Fire,Laura E. Weymouth,2022,2022-11-22,Margaret K. McElderry Books,<NA>,"Ontario, Canada",9781665902700.0,NaN,['young-adult fantasy'],False,False,11,Unknown,0,0,False,False,Canada


In [13]:
## Author Birthplace by continent
continent_dict = {
'Africa' : ['Africa','Zambia','Libya','Sudan','Ethiopia','Madagascar','Morocco','Sudan','Namibia','Niger','Botswana','Liberia','Tunisia','Mauritius',
          'Nyasaland','Sierra Leone','Guinea','Tanzania','Zimbabwe','Uganda','Ghana','Egypt','Kenya','Rwanda','Nigeria','Rhodesia','South Africa',
          ],
'Europe' : ['Cyprus','Marino','Macedonia','Latvia','Holy Roman Empire','Luxembourg','Gibraltar','Bulgaria','Ukraine','Serbia','Bailiwack of Jersey',
          'Denmark','Norway','Iceland','Czech Republic','Greece','Hungary','Finland','Spain','Romania','Malta','Belgium','Poland','Portugal',
          'Switzerland','Yugoslavia','Austria','Sweden','Italy','Netherlands','France','Russia','Germany','Ireland','UK'],
'South America' : ['Suriname','Uruguay','Bolivia','Colombia','Guyana','Chile','Ecuador','Peru','Argentina','Brazil','Venezuela'],
'Central/North America' : ['USA','Honduras','Bahamas','Guatemala','Barbados','Cayman Islands','Croatia','Grenada','Panama','Jamaica','Bermuda',
                         'Haiti','Trinidad and Tobago','Cuba','West Indies','Mexico','Dominican Republic','Canada'],
'Asia' : ['Iraq','East Indies','Myanmar','Philippines','Brunei','Macau','Kuwait','Nepal','United Arab Emirates','Bahrain','Vietnam','Laos','Arabia',
        'Pakistan','Palestine','Sri Lanka','Bangladesh','Persia','Turkey','Lebanon','Ceylon','Korea','Thailand','Indonesia','Taiwan','Iran',
        'Singapore','Malaysia','Philippines','Hong Kong','China','Israel','Japan','India'],
'Oceania' : ['Australia','New Zealand','Samoa','Marshall Islands','Papua New Guinea','Fiji']
}

def replace_author_birthplace_continent(x):
    if pd.isna(x):
        return 'Unknown'
    for country, birthplace in continent_dict.items():
        if any(word in x for word in birthplace):
            return country
    if x not in replace_dict.items():
        return x.strip().split()[-1]
    return x

In [14]:
books = books.assign(author_birthplace_continent=books['author_birthplace'].apply(replace_author_birthplace_continent))
books.sample(15)

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,...,hugo,locus,month_of_publication,Author_Age_at_Publication,Hugo_Awards_Previously,Locus_Awards_Previously,Hugo_Nominee_Before,Locus_Nominee_Before,author_birthplace_country,author_birthplace_continent
12498,987037,Time Trap,Jean M. Favors,1984,1984-00-00,Parachute Press / Scholastic,1938,NaN,059033168X,NaN,...,False,False,Unknown,46,0,0,False,False,Unknown,Unknown
83931,1797758,Unbreakable,W. C. Bauers,2015,2015-01-13,Tor,<NA>,NaN,9780765375421.0,NaN,...,False,False,01,Unknown,0,0,False,False,Unknown,Unknown
137800,3164673,Trapped in the Dark,Jada Fisher,2023,2023-03-28,Fairfield Publishing,<NA>,NaN,NaN,NaN,...,False,False,03,Unknown,0,0,False,False,Unknown,Unknown
67241,1404424,Embrace of the Damned,Anya Bast,2012,2012-05-01,Berkley Sensation,1972,"Minnesota, USA",9780425247969.0,NaN,...,False,False,05,40,0,0,False,False,USA,Central/North America
474,21806,The Domes of Mars,Patrick Moore,1956,1956-00-00,Burke,1923,"Pinner, Middlesex, England, UK",NaN,NaN,...,False,False,Unknown,33,0,0,False,False,UK,Europe
107549,2442033,Shadowblood Heir,J. S. Morin,2018,2018-02-20,Magical Scrivener Press,1977,"New Hampshire, USA",9781942642220.0,NaN,...,False,False,02,41,0,0,False,False,USA,Central/North America
31383,21831,Fortune's Stroke,"David Drake, Eric Flint",2000,2000-06-00,Baen Books,1945,"Dubuque, Iowa, USA",0671319981,EVIL FROM BEYOND TIME RULES THE GREATEST EMPIR...,...,False,False,06,55,0,0,False,False,USA,Central/North America
93060,2049054,Native Wind,A. M. Burns,2016,2016-07-19,DSP Publications,<NA>,NaN,9781634765527.0,NaN,...,False,False,07,Unknown,0,0,False,False,Unknown,Unknown
134231,2993948,Lover Arisen,J. R. Ward,2022,2022-04-05,Gallery Books,1969,"Boston, Massachusetts, USA",9781982179991.0,"Possessed by the demon Devina, Balthazar is on...",...,False,False,04,53,0,0,False,False,USA,Central/North America
91291,2174172,Reckoning in the Void,J. T. Williams,2016,2016-01-14,J. T. Williams,<NA>,NaN,NaN,NaN,...,False,False,01,Unknown,0,0,False,False,Unknown,Unknown


In [ ]:
books.reset_index()
books.to_csv("data_with_author_and_awards.csv", index = False)